# Image evaluation and selection strategies

Inspect classifier measurements, compare selection strategies on the same candidate grid, and configure a short automated run.

ImageNet is the training dataset/category set. The frozen **ResNet-18 trained on ImageNet-1K** returns 1,000 class probabilities per image. A **selection strategy** owns any evaluation, scoring and final choice: random selection needs no classifier, while both ImageNet modes share maximum-class scoring and differ only in the probability of a random choice.

Run from the project root with:

```sh
uv sync --locked --extra imagenet
uv run --extra imagenet jupyter lab notebooks/02_image_evaluation.ipynb
```

This notebook includes classifier inspection, so it requires the ImageNet extra. The first load downloads about 45 MB into `.cache/imagenet`; later loads reuse it. Pure random runs through Python or the CLI need no Torch. No training occurs. This notebook is independent of notebook 01.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from html import escape
import random
import time

import numpy as np
from IPython.display import display
import ipywidgets as widgets

from automated_picbreeder.cppn import load_config, new_genome, render, png_bytes
from automated_picbreeder.evaluation import Evaluation, ImageEvaluator, validate_image
from automated_picbreeder.imagenet import ImageNetEvaluator

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
from automated_picbreeder.experiment import ExperimentSettings, run_experiment
from automated_picbreeder.selection_strategies import RandomSelectionStrategy, ImageNetSelectionStrategy


## 1. Render a fixed set of candidates

Use nine initial candidates with seed 7 (set the human playground to seed 7 to match). The CPPN produces HSB values, which the shared renderer converts to uint8 RGB arrays. Keeping the candidates fixed lets us compare evaluators on the same inputs.

Later, images could come from selected or saved genomes. The evaluator only sees images; it has no dependency on CPPN or NEAT code.

In [ ]:
config = load_config()
rng = random.Random(7)
genomes = [new_genome(i, config, rng) for i in range(9)]
images = [render(genome, config, size=96) for genome in genomes]
print(f"{len(images)} images; shape {images[0].shape}; dtype {images[0].dtype}")

## 2. Load the frozen classifier

`ImageNetEvaluator` remains available for standalone inspection. Its `evaluate(images) -> Evaluation` method returns the image-by-measurement matrix, column names and provenance; it does not make a selection. The experiment runner takes a complete selection strategy, which can reuse this loaded evaluator internally.

We choose `IMAGENET1K_V1` explicitly and use its official preprocessing. CPU is the default. Other supported Torchvision ImageNet models can be selected with `model_name` and a concrete `weights` version. Both greedy and epsilon-greedy ImageNet selection below share this one loaded classifier.


In [ ]:
classifier = ImageNetEvaluator(
    model_name="resnet18",
    weights="IMAGENET1K_V1",
    device="cpu",
    batch_size=16,
    cache_dir=ROOT / ".cache" / "imagenet",
)
evaluator: ImageEvaluator = classifier
classifier.describe()

## 3. Look at the actual classifier input

The network receives RGB images converted from our CPPN’s HSB outputs. Grayscale inputs are also supported by repeating brightness across three channels. The official transform resizes the shorter side to 256 pixels, takes a **central 224 × 224 crop**, rescales to [0, 1], and applies fixed channel normalization.

This removes part of the border. It matters for CPPNs: the classifier does not assess the entire original image. The preview below reverses only channel normalization so you can see the crop. The returned tensor is the normalized input actually used by the model.

In [ ]:
tensor = classifier.preprocess(images[2])
preprocessing = classifier.describe()["preprocessing"]
mean = np.array(preprocessing["mean"])
std = np.array(preprocessing["std"])
crop = tensor.numpy().transpose(1, 2, 0) * std + mean
crop = np.rint(np.clip(crop, 0, 1) * 255).astype(np.uint8)
display(widgets.HBox([
    widgets.VBox([widgets.Label("Original"), widgets.Image(value=png_bytes(images[2]), width=224, height=224)]),
    widgets.VBox([widgets.Label("Classifier crop"), widgets.Image(value=png_bytes(crop), width=224, height=224)]),
]))

## 4. Score every image against every class

`values[i, j]` is image i's score for class index j. Names are display labels; some labels repeat, so use the **column index** to identify a class.

We show just the three largest scores per image, but keep the full 9 × 1,000 matrix for inspection and decision records. These are relative softmax scores, not a calibrated probability that an abstract pattern depicts a real object, and not a measure of naturalness or interestingness.

In [ ]:
started = time.perf_counter()
result = evaluator.evaluate(images)
print(f"Scored {len(images)} images in {time.perf_counter() - started:.2f}s")
print("Full score matrix:", result.values.shape)
print("Row sums:", result.values.sum(axis=1))


def show_scores(images, evaluation):
    """Display measurements without making a selection decision."""
    cards = []
    for i, (image, values) in enumerate(zip(images, evaluation.values)):
        top = np.argsort(-values, kind="stable")[:3]
        rows = "<br>".join(
            f"{escape(evaluation.names[j])} [{j}]: {values[j]:.4f}" for j in top
        )
        cards.append(widgets.VBox([
            widgets.Label(f"Candidate {i}"),
            widgets.Image(value=png_bytes(image), width=150, height=150),
            widgets.HTML(rows),
        ], layout=widgets.Layout(padding="8px")))
    display(widgets.GridBox(cards, layout=widgets.Layout(
        grid_template_columns="repeat(3, minmax(180px, 1fr))",
        grid_gap="8px", width="100%", max_width="850px",
    )))


show_scores(images, result)

## 5. Replace the evaluator

Here is a complete alternative with one column: pixel contrast (standard deviation after division by 255). It implements the same interface without inheriting a base class or knowing about ImageNet.

**Contrast is only an interface demonstration.** It is not a validated measure of interestingness, and optimising it can reward unhelpful patterns. Notice that measurements need not sum to 1 or even be probabilities. A selection strategy must decide how to use them. Changing the complete selection strategy is how experiments are configured; this helper only demonstrates another measurement.

In [ ]:
class ContrastEvaluator:
    def evaluate(self, images):
        for image in images:
            validate_image(image)
        values = np.array([image.astype(float).std() / 255 for image in images]).reshape(-1, 1)
        return Evaluation(
            values=values,
            names=("pixel contrast",),
            metadata={"evaluator": "pixel_contrast", "definition": "std of all uint8 channel values / 255"},
        )


alternative: ImageEvaluator = ContrastEvaluator()
contrast_result = alternative.evaluate(images)
show_scores(images, contrast_result)

## 6. Compare individual measurements

For inspection, choose a single class index and compare its preferred candidate with the highest-contrast candidate. Change `class_index` to any value in [0, 999]. This illustrates how different measurements prefer different images.

The ImageNet selection strategy below uses the highest probability **across all classes** for each candidate. It does not fix a target class or maintain class-specific champions.


In [ ]:
class_index = int(result.values[0].argmax())  # Change to any ImageNet class index.
classifier_choice = int(result.values[:, class_index].argmax())
contrast_choice = int(contrast_result.values[:, 0].argmax())
print(f"Highest score for {result.names[class_index]} [{class_index}]: candidate {classifier_choice}")
print(f"Highest contrast: candidate {contrast_choice}")
display(widgets.HBox([
    widgets.VBox([widgets.Label("Chosen class"), widgets.Image(value=png_bytes(images[classifier_choice]), width=192, height=192)]),
    widgets.VBox([widgets.Label("Contrast"), widgets.Image(value=png_bytes(images[contrast_choice]), width=192, height=192)]),
]))

## 7. Compare selection strategies on the same grid

Each selection strategy receives these same nine images and returns a `SelectionDecision`: a zero-based position, a mode, and optional measurements and preference scores. The runner will later record those details. ImageNet scoring is the maximum class probability per candidate; random selection has no scores.

The two ImageNet configurations share `classifier`. Each call gets a fresh RNG with the same seed, so the comparison is repeatable without consuming the breeding RNG. A 10% exploration probability does not guarantee a different choice on this particular grid. The mode records whether the choice was greedy or random, even if exploration selected the greedy winner.


In [ ]:
selection_strategies = {
    "random": RandomSelectionStrategy(),
    "imagenet": ImageNetSelectionStrategy(evaluator=classifier),
    "imagenet epsilon=0.1": ImageNetSelectionStrategy(epsilon=0.1, evaluator=classifier),
}
comparison_seed = 7
choices = {}
cards = []
for name, selection_strategy in selection_strategies.items():
    decision = selection_strategy.choose(images, rng=random.Random(comparison_seed))
    choices[name] = decision
    score = "unscored" if decision.scores is None else f"score {decision.scores[decision.position]:.6f}"
    print(f"{name}: position {decision.position}, {decision.mode}, {score}")
    cards.append(widgets.VBox([
        widgets.Label(name),
        widgets.Image(value=png_bytes(images[decision.position]), width=192, height=192),
        widgets.Label(f"position {decision.position} | {decision.mode} | {score}"),
    ]))
display(widgets.HBox(cards))


## 8. Run a short breeding experiment

Choose one entry from `selection_strategies`. This cell starts a fresh run with seed 7, makes three decisions, and saves into a new timestamped directory. It does not continue the measurement grid's RNG stream. All strategies use the same rendering and mutation implementation as notebook 01.

Three decisions present 27 candidates and generate 25 genomes. Pure random selection makes zero classifier evaluations; ImageNet selection makes 27, including exploratory turns. Change `steps` for longer runs. `selection_seed` can be supplied in `ExperimentSettings`; otherwise the runner derives it separately from `seed` and saves the resolved value. Matching seeds and settings does not keep trajectories identical after strategies choose different parents.


In [ ]:
selected_strategy = selection_strategies["random"]  # Or "imagenet" / "imagenet epsilon=0.1".
settings = ExperimentSettings(seed=7, steps=3, size=96)
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S-%fZ")
output_dir = ROOT / "runs" / f"notebook-selection-{stamp}"
summary = run_experiment(
    selection_strategy=selected_strategy,
    output_dir=output_dir,
    settings=settings,
)
print(f"Saved session: {output_dir / 'session.json'}")
summary


## Inspect the result

Open `grids/` to inspect every choice and `session.json` for the complete record. ImageNet decisions retain the full class-probability matrix and one maximum-class preference score per candidate. Random choices are marked as random, even when they happen to pick the highest score. Pure random runs have null evaluation and scores. All generated genomes, images and ancestry are saved, including rejected alternatives.

Notebook 03 can load the selected genome or any other saved candidate for interventions. Checkpoints are audit records, not resumable evolution sessions. The representation assessment and main experimental protocol are still unfinished; classifier scores and visually interesting outputs do not demonstrate UFR.

For the command-line equivalents, see the [README](../README.md#automated-selection-experiments). To add a different notion of interest later, implement a new `SelectionStrategy` owning its evaluation, scoring and choice. The runner and breeding loop need not know how that choice was made.

Restart the kernel after changing imported modules. References: [Torchvision ResNet-18](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html), [model/checkpoint API](https://docs.pytorch.org/vision/stable/models.html), and the [experiment brief](../docs/experiment-brief.md).
